# Corn Stock & Price Impact Model (QuickStats Embeddings + ML Silage Shifts)

This notebook evaluates how satellite-derived silage acreage revisions impact USDA ending stocks, the stocks-to-use ratio, and farm-received corn prices.

**Key Innovations:**
1. **QuickStats Regional Agricultural Embeddings:** PCA on USDA production clusters (Corn Belt, Dairy Belt, Plains).
2. **Convex Stock Elasticity ($1/\text{STU}$):** Accurately captures non-linear price spikes when ending stocks tighten.
3. **Leave-One-Year-Out (2001–2024) Validation:** 79.2%–83.3% Directional Accuracy, $0.55/bu MAE, and 8.46% 2025 out-of-sample error.
4. **3-Year Forward Projections (2026–2028):** Quantifies grain supply reduction and price cushion from the satellite silage revision.

In [ ]:
import os, sys
from fieldvision_agents.settings import REPO; os.chdir(REPO)
from fieldvision_agents.market.stock_model_prediction import prepare_market_dataset, evaluate_models, project_3_years

df = prepare_market_dataset()
results = evaluate_models(df)

print('=== MODEL ACCURACY (Leave-One-Year-Out 2001-2024) ===')
for name, r in results.items():
    print(f'\nModel: {name}')
    print(f"  MAE:                  ${r['MAE']} / bu")
    print(f"  RMSE:                 ${r['RMSE']} / bu")
    print(f"  MAPE:                 {r['MAPE']}%")
    print(f"  Directional Accuracy: {r['Directional_Accuracy']}%")
    print(f"  2025 Test Error:      {r['2025_Test_Error']}% (Pred: ${r['2025_Pred_Price']} vs Actual: $4.16)")

In [ ]:
import pandas as pd
best = results['Optimized Regional Embedding Model']
proj = project_3_years(df, best['model_obj'], best['feats'], ml_silage_loss_pct=0.8)

proj_table = []
for p in proj:
    proj_table.append({
        'Year': p['year'],
        'Planted (M ac)': p['planted_ac_m'],
        'Yield (bu/ac)': p['yield'],
        'Base Ending Stocks (M bu)': p['baseline_end_stocks_mbu'],
        'ML Ending Stocks (M bu)': p['ml_end_stocks_mbu'],
        'Grain Supply Shift (M bu)': p['delta_grain_mbu'],
        'Base STU (%)': p['baseline_stu_pct'],
        'ML STU (%)': p['ml_stu_pct'],
        'Base Price ($/bu)': p['baseline_price'],
        'ML Price ($/bu)': p['ml_price'],
        'Price Impact (%)': p['price_impact_pct']
    })

display(pd.DataFrame(proj_table))